
# 01/ Test DESC SN Ia metric SNNSNMMetrics  

try with `sne_nside = 16`

This metric is contributed by Philipe Gris. The metric creates a SN population at a range of redshifts, looks for detections pre and post peak, and then evaluates the number of SN that constitute a complete sample out to a redshift value defined by how well the color covariance can be defined. 

- author of corrections : Sylvie Dagoret-Campagne
- creation date : 2026-08-05
- copied and adapted (corrected) from https://github.com/lsst/rubin_sim_notebooks/tree/main/maf/science/testSNIa.ipynb

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import healpy as hp
import pandas as pd


import rubin_sim
from rubin_sim import maf
from rubin_sim.data import get_baseline

## Configuration

In [ ]:
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
baseline_file = get_baseline()
# opsdb = maf.db.OpsimDatabase(baseline_file)
# opsdb =  maf.db.add_run_to_database(baseline_file)
run_name = os.path.split(baseline_file)[-1].replace(".db", "")

print(run_name)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="01_maf_testSNIa_", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
out_dir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=out_dir)

## SNNSNMMetrics

### Check info on SNNSNMMetrics 

In [ ]:
# to view the signature of the Metrics class
%pinfo maf.SNNSNMetric

In [ ]:
# to view the code of the metrics
# %psource maf.SNNSNMetric

### Define the metrics and group bundle

In [ ]:
#  Set up to time it at one point in the sky

plot_dict = {"percentileClip": 95.0, "nTicks": 5}

sne_nside = 16
dustmap = maf.DustMap(nside=sne_nside)

# summary metrics
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]

# Healpix slicer
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)

# SNNSNMetric metrics
metric = maf.SNNSNMetric(
    n_bef=3,
    n_aft=8,
    coadd_night=True,
    add_dust=False,
    hard_dust_cut=0.25,
    zmin=0.1,
    zmax=0.5,
    z_step=0.03,
    daymax_step=3.0,
    zlim_coeff=0.95,
    gamma_name="gamma_WFD.hdf5",
    verbose=False,
)

# Bundle
bundle = maf.MetricBundle(
    metric,
    slicer,
    None,
    plot_dict=plot_dict,
    maps_list=[dustmap],
    summary_metrics=sn_summary,
    run_name=run_name,
)

# Bundle group
bg = maf.MetricBundleGroup({"sn": bundle}, baseline_file, out_dir, resultsDb)

In [ ]:
%pinfo bg.get_data

In [ ]:
# Behind the scenes stuff to get the simulated data and set up the slicer so we can test ONE point
# constraint
bg.set_current("")
# Query the data from the database.
bg.get_data("")
# retreive data
sim_data = bg.sim_data
bundle.slicer.setup_slicer(sim_data)
# Add the dust extinction
slicer.slice_points = dustmap.run(slicer.slice_points)

In [ ]:
# Find a good spot on the sky with lots of visits
len_visits = []
for s in bundle.slicer:
    len_visits.append(len(s["idxs"]))
lenvisits = np.array(len_visits)
x = np.where(lenvisits == np.max(lenvisits))[0][0]
print(f"x = {x}")
print("bundle.slicer[x] = ", bundle.slicer[x])

In [ ]:
# sid = 890 - a point with a single usable season
sid = 2593
bundle.slicer[sid]

### Run only one slice

In [ ]:
%%time

metric.run(sim_data[bundle.slicer[sid]["idxs"]], bundle.slicer[sid]["slice_point"])

In [ ]:
print(
    len(slicer) * 9 / 60 / 60 / 2, "hrs best guess metric run time"
)  # /2 because maybe half sky doesn't have visits

In [ ]:
sim_data

In [ ]:
# Take a moment and create some metric test data
s = sim_data[bundle.slicer[2593]["idxs"]]
s.sort(order="observationStartMJD")
dd = np.where(np.diff(s["night"]) > 80)[0] + 1
o = s[dd[3] : dd[4]]
n, b, p = plt.hist(o["night"], bins=100)
o2 = o[np.where(o["scheduler_note"] != "DD:ELAISS1")]
n, b, p = plt.hist(o2["night"], bins=100)
o3 = o2.copy()
o3["fiveSigmaDepth"] = o3["fiveSigmaDepth"] - 4
len(o), len(o2)
o4 = o2.copy()
o4["numExposures"] = 1
o4["visitExposureTime"] = 20
o5 = o2.copy()
o5["numExposures"] = 1

In [ ]:
pd.DataFrame(sim_data[bundle.slicer[2593]["idxs"]]).to_hdf("test_simData.hdf", key="dense_pointing", mode="w")
pd.DataFrame(sim_data[bundle.slicer[890]["idxs"]]).to_hdf("test_simData.hdf", key="sparse_pointing", mode="a")
pd.DataFrame(o2).to_hdf("test_simData.hdf", key="one_season_noDD", mode="a")
pd.DataFrame(o).to_hdf("test_simData.hdf", key="one_season_wDD", mode="a")
pd.DataFrame(o3).to_hdf("test_simData.hdf", key="one_season_shallow", mode="a")
pd.DataFrame(o4).to_hdf("test_simData.hdf", key="one_season_singleExp_20", mode="a")
pd.DataFrame(o5).to_hdf("test_simData.hdf", key="one_season_singleExp_30", mode="a")

### Run all slices

In [ ]:
%%time 
# -- this will run the whole sky, so could be close to the estimate above
bg.run_all()

In [ ]:
bg.plot_all(closefigs=False)

In [ ]:
bundle.metric_values.compressed()[0]

In [ ]:
# The 'reduce' values of the metric got stored in the bundle dict in the bungle group
# (which is why we usually set this as a dictionary outside of the metricBundleGroup call .. whoops.
bg.bundle_dict

In [ ]:
# The nSN and zlim values are pulled out in those reduce methods, into their own bundles.
bdict = bg.bundle_dict
bdict["SNNSNMetric_reducen_sn"].metric_values.compressed()[0:10]

In [ ]:
bdict["SNNSNMetric_reducezlim"].metric_values.compressed()[0:10]

In [ ]:
for k in bdict:
    print(k, bdict[k].summary_values)

In [ ]:
# run_name = 'rolling_bulge_ns2_rw0.9_v2.0_10yrs'
run_name = "ddf_sd_v5.3.0_10yrs"
opsdb = os.path.join(path_topdir, f"{run_name}.db")

plot_dict = {"percentileClip": 95.0, "nTicks": 5}

sne_nside = 16
dustmap = maf.DustMap(nside=16)
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)
metric = maf.SNNSNMetric(
    n_bef=3,
    n_aft=8,
    coadd_night=True,
    add_dust=False,
    hard_dust_cut=0.25,
    zmin=0,
    zmax=1.2,
    z_step=0.03,
    daymax_step=3.0,
    zlim_coeff=0.95,
    gamma_name="gamma_WFD.hdf5",
    verbose=False,
)
bundle2 = maf.MetricBundle(
    metric,
    slicer,
    None,
    run_name=run_name,
    plot_dict=plot_dict,
    maps_list=[dustmap],
    summary_metrics=sn_summary,
)


bg = maf.MetricBundleGroup({"sn": bundle2}, opsdb, out_dir, resultsDb)

In [ ]:
%%time
bg.run_all()

In [ ]:
bdict = bg.bundle_dict.copy()
for k in bdict:
    print(k, bdict[k].summary_values)

In [ ]:
bg.plot_all(closefigs=False)